In [1]:
%env WAVE_CACHE_ON=0
%env PYTHONPATH=/home/tim/src/iree/build/compiler/bindings/python:/home/tim/src/iree/build/runtime/bindings/python
%env IREE_SAVE_TEMPS=/home/tim/src/iree-turbine/dump

env: WAVE_CACHE_ON=0
env: PYTHONPATH=/home/tim/src/iree/build/compiler/bindings/python:/home/tim/src/iree/build/runtime/bindings/python
env: IREE_SAVE_TEMPS=/home/tim/src/iree-turbine/dump


In [ ]:
import iree.turbine.kernel.lang as tkl
from iree.turbine.kernel.lang.kernel_buffer import MemoryLayout
import iree.turbine.kernel.wave as tkw
from iree.turbine.kernel.lang.global_symbols import *
from iree.turbine.kernel.wave.utils.general_utils import (
    get_default_scheduling_params,
)
from iree.turbine.kernel.wave.scheduling.schedule import SchedulingType
from iree.turbine.kernel.wave.compile import WaveCompileOptions, wave_compile


from iree.turbine.kernel.wave.constraints import GenericDot, MMAOperand, MMAType, ScaledMMAType

# Add test shapes for validation and performance testing.
SHAPE = (4096, 4096, 4096)
SCHEDULING_TYPE = SchedulingType.NONE # SchedulingType.PREFETCH, SchedulingType.MODULO, SchedulingType.MODULO_MULTI_BUFFERED,
DYNAMIC_DIMS = False
MFMA_VARIANT = ScaledMMAType.F32_16x16x128_F8F6F4 # F32_16x16x16_F16 # MMAType.F32_32x32x8_F16
DUMP_GENERATED_MIR = True

def testGemm(
    shape: tuple[int] = SHAPE,
    enable_scheduling: SchedulingType = SCHEDULING_TYPE,
    dynamic_dims: bool = DYNAMIC_DIMS,
    mfma_variant: MMAType = MFMA_VARIANT
):
    # Input sizes
    M = tkl.sym.M
    N = tkl.sym.N
    K = tkl.sym.K
    # Workgroup tile sizes
    BLOCK_M = tkl.sym.BLOCK_M
    BLOCK_N = tkl.sym.BLOCK_N
    BLOCK_K = tkl.sym.BLOCK_K
    # Address space (for GPU, shared(1) or global(0))
    ADDRESS_SPACE = tkl.sym.ADDRESS_SPACE

    # Expose user-constraints
    constraints: list[tkw.Constraint] = []
    constraints += [tkw.WorkgroupConstraint(M, BLOCK_M, 0)]
    constraints += [tkw.WorkgroupConstraint(N, BLOCK_N, 1)]
    constraints += [tkw.TilingConstraint(K, BLOCK_K)]
    constraints += [tkw.WaveConstraint(M, BLOCK_M / 2)]
    constraints += [tkw.WaveConstraint(N, BLOCK_N / 2)]

    constraints += [
        tkw.HardwareConstraint(
            threads_per_wave=64, waves_per_block=(2, 2, 1), mma_type=mfma_variant
        )
    ]

    # Wave-level micro-kernel.
    # Since warps are not directly addressable, there is no
    # explicit notion of a warp id (like a workgroup or thread id).
    # This kernel uses the input sizes M, N, K throughout, as the tiling
    # and data movement strategy is determined during the compilation process.
    # These can be influenced by introducing constraints.
    @tkw.wave(constraints)
    def gemm(
        a: tkl.Memory[M, K, ADDRESS_SPACE, tkl.f8e5m2],
        a_scale: tkl.Memory[M, K, ADDRESS_SPACE, tkl.f8e8m0fnu, MemoryLayout(shape=(M, K/32))],
        b: tkl.Memory[N, K, ADDRESS_SPACE, tkl.f8e5m2],
        b_scale: tkl.Memory[N, K, ADDRESS_SPACE, tkl.f8e8m0fnu, MemoryLayout(shape=(N, K/32))],
        c: tkl.Memory[M, N, GLOBAL_ADDRESS_SPACE, tkl.f32],
    ):
        c_reg = tkl.Register[M, N, tkl.f32](0.0)

        @tkw.iterate(K, init_args=[c_reg])
        def repeat(acc: tkl.Register[M, N, tkl.f32]) -> tkl.Register[M, N, tkl.f32]:
            a_reg = tkw.read(a)
            a_scale_reg = tkw.read(a_scale)
            b_reg = tkw.read(b)
            b_scale_reg = tkw.read(b_scale)
            acc = tkw.scaled_mma(a_reg, a_scale_reg, b_reg, b_scale_reg, acc)
            return acc

        tkw.write(repeat, c)

    hyperparams = {
        ADDRESS_SPACE: SHARED_ADDRESS_SPACE,
        BLOCK_M: 32,
        BLOCK_N: 32,
        BLOCK_K: 128,
        M: shape[0],
        N: shape[1],
        K: shape[2],
    }
    hyperparams.update(get_default_scheduling_params())

    dynamic_symbols = []
    dynamic_symbols_map = {}
    if dynamic_dims:
        dynamic_symbols_map[M] = hyperparams[M]
        dynamic_symbols_map[N] = hyperparams[N]
        dynamic_symbols_map[K] = hyperparams[K]
        dynamic_symbols.append(M)
        dynamic_symbols.append(N)
        dynamic_symbols.append(K)
        del hyperparams[M]
        del hyperparams[N]
        del hyperparams[K]

    options = WaveCompileOptions(
        subs=hyperparams,
        canonicalize=True,
        schedule=enable_scheduling,
        use_scheduling_barriers=False,
        dynamic_symbols=dynamic_symbols,
        dynamic_symbols_map=dynamic_symbols_map,
        backend = "rocm",
        target = "gfx950",
        print_ir_after_all = False,
        dump_intermediates = "dump"
    )
    
    gemm = wave_compile(options, gemm)

    if DUMP_GENERATED_MIR:
        filename = f"wave_gemm_mma_{'x'.join(map(str, shape))}.mlir"
        with open(filename, "w") as f:
            f.write(gemm.asm)

    return gemm.asm

testGemm()

/home/tim/src/iree-turbine/.venv/lib/python3.11/site-packages/setuptools/version.py:1: UserWarning: Module iree was already imported from None, but /home/tim/src/iree-turbine is being added to sys.path
  import pkg_resources


NameError: name 'KScale' is not defined

In [ ]:
import torch

def from_float(values):
    result = torch.empty_like(values, dtype=torch.uint8)

    is_invalid = torch.isnan(values) | torch.isinf(values) | (values <= 0)
    result[is_invalid] = 255

    valid_values = values[~is_invalid]
    e = torch.floor(torch.log2(valid_values))
    e_biased = e + 127
    e_biased_int = e_biased.type(torch.int32)
    e_biased_clamped = torch.clamp(e_biased_int, 0, 254)
    result[~is_invalid] = e_biased_clamped.type(torch.uint8)

    return result

a = torch.eye(4096, dtype=torch.float32).to(torch.float8_e5m2)
scales_a = from_float(torch.ones((4096, 4096), dtype=torch.float32))

b = torch.eye(4096, dtype=torch.float32).to(torch.float8_e5m2)
scales_b = from_float(torch.ones((4096, 4096), dtype=torch.float32))


tensor([[127, 127, 127,  ..., 127, 127, 127],
        [127, 127, 127,  ..., 127, 127, 127],
        [127, 127, 127,  ..., 127, 127, 127],
        ...,
        [127, 127, 127,  ..., 127, 127, 127],
        [127, 127, 127,  ..., 127, 127, 127],
        [127, 127, 127,  ..., 127, 127, 127]], dtype=torch.uint8)
